# Exploratory S4b/S7a notebook

> Historical interactive interface. The canonical matched study is implemented by `run_micro_ablations.py`.

This notebook demonstrates two gold-informed diagnostic variants:
- S4b applies weighted RRF to the original query and one selected rewrite;
- S7a calls canonical `retriever_s7.s7_rank_controls(...)` with a selected reranking-query override.

These lettered variants are non-deployable diagnostics, not additional systems in the S1--S7 retrieval ladder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================================
# Imports
# ==========================================================
import os
import sys
import importlib
import math, json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:.4f}".format


In [ ]:
# ==========================================================
# 0) CONFIG — GOLD-INFORMED DIAGNOSTICS (S4b + S7a)
# ==========================================================

DRIVE_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"

# Canonical CCS paths (clause-level JSONL)
REV5_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# Gold sets + error bank (match your existing notebook layout)
REV5_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv"
ERROR_BANK_CSV = f"{DRIVE_ROOT}/data/error_bank/error_bank_v1.csv"

# QUR rewrites (rev5 / rev4 / error-bank)
QUR_CSV_REV5 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_error_bank.csv"

# Output directory (must exist)
ABLATION_OUTPUT_DIR = f"{DRIVE_ROOT}/experiments/micro_ablations"
BASE_OUT_DIR = ABLATION_OUTPUT_DIR
os.makedirs(ABLATION_OUTPUT_DIR, exist_ok=True)

# Models (match your current experiment settings)
DENSE_MODEL_ID = "intfloat/e5-small-v2"
RERANKER_MODEL_ID = "BAAI/bge-reranker-base"

# Micro-ablation hyperparams
CANDIDATE_SET_SIZE = 50
RRF_K_DEFAULT = 60

REWRITE_WEIGHT = 0.25
REWRITE_JACCARD_MIN = 0.15
MAX_REWRITES = 3

RERANK_ALPHA = 0.65
RERANK_APPLY_MIN_MARGIN_RATIO = 0.15
RERANK_SKIP_ENABLED = True
RERANK_SKIP_MIN_BASE_MARGIN_RATIO = 0.10
RERANK_SKIP_REQUIRE_TOP1_AGREEMENT = False

TOP_N = 10


In [ ]:
# ==========================================================
# IMPORT: Canonical S7 retriever (shared library)
# ==========================================================
src_path = os.path.join(DRIVE_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

def _import_retriever_s7() -> Any:
    try:
        from compliancegpt.retriever import retriever_s7 as mod  # type: ignore
        return importlib.reload(mod)
    except Exception:
        retriever_dir = os.path.join(src_path, "compliancegpt", "retriever")
        if retriever_dir not in sys.path:
            sys.path.insert(0, retriever_dir)
        import retriever_s7 as mod  # type: ignore
        return importlib.reload(mod)

retriever_s7 = _import_retriever_s7()
print(f"[INFO] Imported retriever_s7 from: {retriever_s7.__file__}")

ComplianceGPTRetriever = retriever_s7.ComplianceGPTRetriever
RetrievalConfig = retriever_s7.RetrievalConfig
s7_rank_controls = retriever_s7.s7_rank_controls

# Utilities
normalize_control_id = retriever_s7.normalize_control_id
normalize_text = retriever_s7.normalize_text
tokenize = retriever_s7.tokenize
build_query_variants = retriever_s7.build_query_variants


In [ ]:
# ==========================================================
# 1) LOAD DATA + BUILD RETRIEVERS (REV4 + REV5)
# ==========================================================
import inspect
import dataclasses

print("--- Loading Gold Sets ---")
rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1")
rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1")

print("--- Loading Error Bank ---")
error_bank_df = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1")

print("--- Loading QUR rewrites ---")
qur_rev5_df = pd.read_csv(QUR_CSV_REV5, encoding="ISO-8859-1")
qur_rev4_df = pd.read_csv(QUR_CSV_REV4, encoding="ISO-8859-1")
qur_errb_df = pd.read_csv(QUR_CSV_ERRB, encoding="ISO-8859-1")

def _assert_s7_dependencies() -> None:
    # Required deps for dense embeddings + rerank model inference.
    missing: List[str] = []
    try:
        import torch  # noqa: F401
    except Exception:
        missing.append("torch")
    try:
        from sentence_transformers import SentenceTransformer, CrossEncoder  # noqa: F401
    except Exception:
        missing.append("sentence-transformers")

    if missing:
        raise RuntimeError(
            "S7 requires torch + sentence-transformers for dense retrieval + reranking. "
            f"Missing: {missing}"
        )

    # faiss may be optional depending on retriever_s7.py implementation.
    try:
        import faiss  # type: ignore  # noqa: F401
        print("[INFO] faiss is available.")
    except Exception:
        print(
            "[WARN] faiss is not available in this runtime. "
            "If your imported retriever_s7.py hard-requires faiss, retriever initialization will fail. "
            "If it supports a non-faiss fallback, it should continue (potentially slower)."
        )

def _build_retrieval_config(overrides: Dict[str, Any]) -> Tuple[Any, List[str]]:
    # Dataclass path (preferred)
    try:
        if dataclasses.is_dataclass(RetrievalConfig):
            allowed = {f.name for f in dataclasses.fields(RetrievalConfig)}
            filtered = {k: v for k, v in overrides.items() if k in allowed}
            ignored = [k for k in overrides.keys() if k not in allowed]
            return RetrievalConfig(**filtered), ignored
    except Exception:
        pass

    # Signature path
    try:
        sig = inspect.signature(RetrievalConfig)
        allowed = set(sig.parameters.keys())
        filtered = {k: v for k, v in overrides.items() if k in allowed}
        ignored = [k for k in overrides.keys() if k not in allowed]
        return RetrievalConfig(**filtered), ignored
    except Exception:
        pass

    # Last resort: default instance + setattr (may fail if frozen)
    cfg = RetrievalConfig()
    ignored: List[str] = []
    for k, v in overrides.items():
        if hasattr(cfg, k):
            try:
                setattr(cfg, k, v)
            except Exception:
                ignored.append(k)
        else:
            ignored.append(k)
    return cfg, ignored

cfg_overrides: Dict[str, Any] = {
    "dense_model_id": DENSE_MODEL_ID,
    "reranker_model_id": RERANKER_MODEL_ID,
    "candidate_set_size": CANDIDATE_SET_SIZE,
    "rrf_k": RRF_K_DEFAULT,
    "max_rewrites": MAX_REWRITES,
    "rewrite_weight": REWRITE_WEIGHT,
    "rewrite_jaccard_min": REWRITE_JACCARD_MIN,
    "rerank_alpha": RERANK_ALPHA,
    "rerank_apply_min_margin_ratio": RERANK_APPLY_MIN_MARGIN_RATIO,
    "rerank_skip_enabled": RERANK_SKIP_ENABLED,
    "rerank_skip_min_base_margin_ratio": RERANK_SKIP_MIN_BASE_MARGIN_RATIO,
    "rerank_skip_require_top1_agreement": RERANK_SKIP_REQUIRE_TOP1_AGREEMENT,
}

base_cfg, ignored_keys = _build_retrieval_config(cfg_overrides)
if ignored_keys:
    print("[WARN] Ignored RetrievalConfig keys (not supported by this retriever_s7.py):", ignored_keys)

print("[INFO] Using RetrievalConfig:", base_cfg)

_assert_s7_dependencies()

print("--- Building Rev5 retriever ---")
try:
    rev5_retriever = ComplianceGPTRetriever(ccs_path=REV5_CATALOG_PATH, config=base_cfg)
except ImportError as e:
    msg = str(e).lower()
    if "faiss" in msg:
        raise RuntimeError(
            "Retriever initialization failed due to missing 'faiss'. "
            "Either install faiss in this runtime, or use a retriever_s7.py that provides a non-faiss fallback."
        ) from e
    raise

print("--- Building Rev4 retriever ---")
try:
    rev4_retriever = ComplianceGPTRetriever(ccs_path=REV4_CATALOG_PATH, config=base_cfg)
except ImportError as e:
    msg = str(e).lower()
    if "faiss" in msg:
        raise RuntimeError(
            "Retriever initialization failed due to missing 'faiss'. "
            "Either install faiss in this runtime, or use a retriever_s7.py that provides a non-faiss fallback."
        ) from e
    raise

print("[INFO] Rev5 clauses:", len(getattr(rev5_retriever, "ids", [])), " Rev4 clauses:", len(getattr(rev4_retriever, "ids", [])))


In [ ]:
# ==========================================================
# 2) GOLD-INFORMED DIAGNOSTIC LOGIC — S4b + S7a
# ==========================================================

def _strict_clean(s: str) -> str:
    return "".join(c.lower() for c in str(s) if c.isalnum())

def _jaccard_overlap(a: str, b: str) -> float:
    a_t = set(tokenize(normalize_text(a)))
    b_t = set(tokenize(normalize_text(b)))
    if not a_t and not b_t:
        return 1.0
    if not a_t or not b_t:
        return 0.0
    return float(len(a_t & b_t)) / float(len(a_t | b_t))

def _get_qur_rewrites(original_query: str, qur_df: pd.DataFrame, *, max_rewrites: int, jaccard_min: float) -> List[str]:
    """
    Supports two common QUR shapes:
      (A) long: columns ['original_query','rewritten_query'] with multiple rows per original
      (B) wide: columns like ['question','rewrite_1','rewrite_2','rewrite_3'] or similar
    Returns only rewrite strings (no original).
    """
    if qur_df is None or qur_df.empty:
        return []

    cols_lower = {str(c).lower(): c for c in qur_df.columns}
    cols = set(cols_lower.keys())

    # Long format
    if "original_query" in cols and "rewritten_query" in cols:
        df = qur_df.copy()
        if "strict_key" not in df.columns:
            df["strict_key"] = df[cols_lower["original_query"]].apply(_strict_clean)
        key = _strict_clean(original_query)
        matches = df[df["strict_key"] == key]
        rewrites: List[str] = []
        for r in matches[cols_lower["rewritten_query"]].tolist():
            r = str(r).strip()
            if len(r) < 6:
                continue
            if _jaccard_overlap(original_query, r) < float(jaccard_min):
                continue
            rewrites.append(r)
        out = list(dict.fromkeys(rewrites))[: int(max_rewrites)]
        return out

    # Wide format
    rewrite_cols = [c for c in qur_df.columns if re.search(r"(rewrite|rewritten)", str(c), flags=re.I)]
    key_cols = [c for c in qur_df.columns if str(c).lower() in {"question", "original_query", "query"}]
    if rewrite_cols and key_cols:
        qcol = key_cols[0]
        df = qur_df.copy()
        df["_strict_key"] = df[qcol].apply(_strict_clean)
        key = _strict_clean(original_query)
        matches = df[df["_strict_key"] == key]
        if matches.empty:
            return []
        row0 = matches.iloc[0]
        rewrites: List[str] = []
        for c in rewrite_cols:
            r = str(row0.get(c, "")).strip()
            if len(r) < 6:
                continue
            if _jaccard_overlap(original_query, r) < float(jaccard_min):
                continue
            rewrites.append(r)
        out = list(dict.fromkeys(rewrites))[: int(max_rewrites)]
        return out

    return []


def _rrf_fuse(lists: List[List[str]], *, k: int, weights: Optional[List[float]] = None) -> List[str]:
    """
    Reciprocal Rank Fusion over ranked lists.
    Returns a single ranked list (best first).
    """
    weights = weights or [1.0] * len(lists)
    scores: Dict[str, float] = {}
    for w, lst in zip(weights, lists):
        for rank, cid in enumerate(lst, start=1):
            cid_n = normalize_control_id(cid).upper()
            if not cid_n:
                continue
            scores[cid_n] = scores.get(cid_n, 0.0) + float(w) * (1.0 / (float(k) + float(rank)))
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked]


def _rank_of(gold: str, ranked: List[str], top_n: int) -> int:
    g = normalize_control_id(gold).upper()
    for i, cid in enumerate(ranked[:top_n], start=1):
        if normalize_control_id(cid).upper() == g:
            return i
    return 0


def choose_best_rewrite_for_gold(
    retriever: Any,
    query: str,
    gold_control_id: str,
    qur_df: pd.DataFrame,
    *,
    max_rewrites: int,
    jaccard_min: float,
    candidate_set_size: int,
    rrf_k: int,
    rewrite_weight: float,
) -> Optional[str]:
    """
    Gold-informed selection: pick the rewrite that yields the best (lowest) gold rank
    under the base (no-rerank) S7 control ranking (BM25 + Dense RRF fusion).
    """
    rewrites = _get_qur_rewrites(query, qur_df, max_rewrites=max_rewrites, jaccard_min=jaccard_min)
    if not rewrites:
        return None

    best: Optional[str] = None
    best_rank: Optional[int] = None

    for rw in rewrites:
        variants = [query, rw]
        ranked, _meta = s7_rank_controls(
            original_query=query,
            variants=variants,
            bm25_retriever=retriever.bm25,
            bm25_control_ids=retriever.bm25_control_ids,
            dense_adapter=retriever.dense_adapter,
            reranker=retriever.reranker,
            reranker_text_by_control=retriever.reranker_text_by_control,
            candidate_set_size=int(candidate_set_size),
            rrf_k=int(rrf_k),
            rewrite_weight=float(rewrite_weight),
            rerank_alpha=float(RERANK_ALPHA),
            rerank_apply_min_margin_ratio=float(RERANK_APPLY_MIN_MARGIN_RATIO),
            # Force "no rerank" for selection determinism
            rerank_skip_enabled=True,
            rerank_skip_min_base_margin_ratio=0.0,
            rerank_skip_require_top1_agreement=False,
        )
        r = _rank_of(gold_control_id, ranked, int(candidate_set_size))
        if r == 0:
            continue
        if best_rank is None or r < best_rank:
            best_rank = r
            best = rw

    return best


def system_s4b_rrf_best(
    retriever: Any,
    query: str,
    gold_control_id: str,
    qur_df: pd.DataFrame,
    *,
    top_n: int,
    rrf_k: int,
    rewrite_weight: float,
    max_rewrites: int,
    jaccard_min: float,
) -> Dict[str, Any]:
    """
    S4b: original query plus the gold-informed selected rewrite (weighted RRF over BM25).
    """
    best_rewrite = choose_best_rewrite_for_gold(
        retriever,
        query,
        gold_control_id,
        qur_df,
        max_rewrites=max_rewrites,
        jaccard_min=jaccard_min,
        candidate_set_size=CANDIDATE_SET_SIZE,
        rrf_k=rrf_k,
        rewrite_weight=rewrite_weight,
    )

    variants = [query] + ([best_rewrite] if best_rewrite else [])
    weights = [1.0] + [float(rewrite_weight)] * (len(variants) - 1)

    lists: List[List[str]] = []
    for v in variants:
        toks = tokenize(normalize_text(v))
        idxs = retriever.bm25.get_top_n(toks, n=int(top_n))
        lists.append([normalize_control_id(retriever.bm25_control_ids[i]) for i in idxs])

    fused = _rrf_fuse(lists, k=int(rrf_k), weights=weights)
    retrieved = fused[: int(top_n)]

    return {
        "retrieved_ids": retrieved,
        "system_meta": {"sys": "S4b", "n_vars": len(variants), "best_rewrite_used": bool(best_rewrite)},
    }


def system_s7a_rerank_best(
    retriever: Any,
    query: str,
    gold_control_id: str,
    qur_df: pd.DataFrame,
    *,
    top_n: int,
    max_rewrites: int,
    jaccard_min: float,
) -> Dict[str, Any]:
    """
    S7a: Candidate generation unchanged (original + filtered rewrites),
         reranker query uses the gold-informed selected rewrite (if any).
    """
    best_rewrite = choose_best_rewrite_for_gold(
        retriever,
        query,
        gold_control_id,
        qur_df,
        max_rewrites=max_rewrites,
        jaccard_min=jaccard_min,
        candidate_set_size=CANDIDATE_SET_SIZE,
        rrf_k=RRF_K_DEFAULT,
        rewrite_weight=REWRITE_WEIGHT,
    )

    rewrite_list = _get_qur_rewrites(query, qur_df, max_rewrites=max_rewrites, jaccard_min=jaccard_min)
    variants = build_query_variants(query, rewrite_list, max_rewrites=max_rewrites, jaccard_min=jaccard_min)

    rerank_query = best_rewrite if best_rewrite else query

    ranked, meta = s7_rank_controls(
        original_query=rerank_query,
        variants=variants,
        bm25_retriever=retriever.bm25,
        bm25_control_ids=retriever.bm25_control_ids,
        dense_adapter=retriever.dense_adapter,
        reranker=retriever.reranker,
        reranker_text_by_control=retriever.reranker_text_by_control,
        candidate_set_size=int(CANDIDATE_SET_SIZE),
        rrf_k=int(RRF_K_DEFAULT),
        rewrite_weight=float(REWRITE_WEIGHT),
        rerank_alpha=float(RERANK_ALPHA),
        rerank_apply_min_margin_ratio=float(RERANK_APPLY_MIN_MARGIN_RATIO),
        rerank_skip_enabled=bool(RERANK_SKIP_ENABLED),
        rerank_skip_min_base_margin_ratio=float(RERANK_SKIP_MIN_BASE_MARGIN_RATIO),
        rerank_skip_require_top1_agreement=bool(RERANK_SKIP_REQUIRE_TOP1_AGREEMENT),
    )

    retrieved = [normalize_control_id(c).upper() for c in ranked][: int(top_n)]
    meta = dict(meta or {})
    meta["sys"] = "S7a"
    meta["best_rewrite_used"] = bool(best_rewrite)
    meta["rerank_query_used"] = rerank_query
    meta["variants_n"] = len(variants)

    return {"retrieved_ids": retrieved, "system_meta": meta}


In [ ]:
# ==========================================================
# 3) EVALUATION (Control-level hit)
# ==========================================================
def _hit_at_k(gold_cid: str, retrieved: List[str], k: int) -> bool:
    g = normalize_control_id(gold_cid).upper()
    for cid in retrieved[:k]:
        if normalize_control_id(cid).upper() == g:
            return True
    return False

def _rank_of_gold(gold_cid: str, retrieved: List[str], k: int) -> Optional[int]:
    g = normalize_control_id(gold_cid).upper()
    for i, cid in enumerate(retrieved[:k], 1):
        if normalize_control_id(cid).upper() == g:
            return i
    return None

def _mrr_at_k(gold_cid: str, retrieved: List[str], k: int) -> float:
    r = _rank_of_gold(gold_cid, retrieved, k)
    return 0.0 if r is None else 1.0 / float(r)

def _ndcg_at_k_single(gold_cid: str, retrieved: List[str], k: int) -> float:
    r = _rank_of_gold(gold_cid, retrieved, k)
    if r is None:
        return 0.0
    return 1.0 / math.log2(r + 1)

def evaluate_dataset(
    tag: str,
    df: pd.DataFrame,
    run_fn,
    *,
    gold_col: str,
    question_col: str = "question",
    dataset_tag: Optional[str] = None,
) -> pd.DataFrame:
    """
    Run a system over a dataset and return a per-query results DataFrame.
    Run a system over a dataset and return a per-query results DataFrame.
    """
    rows = []
    for idx, row in df.iterrows():
        q = str(row[question_col])
        gold = str(row[gold_col])

        res = run_fn(q, gold)
        retrieved = res.get("retrieved_ids", []) or []
        meta = res.get("system_meta", {}) or {}

        rank10 = _rank_of_gold(gold, retrieved, 10)
        rows.append({
            "dataset": dataset_tag or "",
            "question_id": row.get("ID", row.get("id", row.get("question_id", idx))),
            "question": q,
            "gold_control_id": normalize_control_id(gold),
            "rank": int(rank10) if rank10 is not None else 0,
            "is_hit_at_1": _hit_at_k(gold, retrieved, 1),
            "is_hit_at_5": _hit_at_k(gold, retrieved, 5),
            "is_hit_at_10": _hit_at_k(gold, retrieved, 10),
            "mrr@10": _mrr_at_k(gold, retrieved, 10),
            "ndcg@10": _ndcg_at_k_single(gold, retrieved, 10),
            "top_1_hit": normalize_control_id(retrieved[0]) if retrieved else "",
            "retrieved_control_ids": " | ".join([normalize_control_id(x) for x in retrieved[:10]]),
            "system_meta": json.dumps(meta, ensure_ascii=False),
        })

    out_df = pd.DataFrame(rows)

    return out_df


In [ ]:
# ==========================================================
# 4) RUN GOLD-INFORMED DIAGNOSTICS — WRITE EXACTLY TWO CSVs
# ==========================================================

ver_col = "version" if "version" in error_bank_df.columns else ("framework_version" if "framework_version" in error_bank_df.columns else None)
assert ver_col is not None, "Error Bank must contain a 'version' (or 'framework_version') column."

err_rev5 = error_bank_df[error_bank_df[ver_col].astype(str).str.contains("rev5", case=False, na=False)].copy()
err_rev4 = error_bank_df[error_bank_df[ver_col].astype(str).str.contains("rev4", case=False, na=False)].copy()

datasets = [
    ("rev5_gold", rev5_gold, rev5_retriever, qur_rev5_df, "control_id"),
    ("rev4_gold", rev4_gold, rev4_retriever, qur_rev4_df, "control_id"),
    ("error_bank_rev5", err_rev5, rev5_retriever, qur_errb_df, "gold_control_id"),
    ("error_bank_rev4", err_rev4, rev4_retriever, qur_errb_df, "gold_control_id"),
]

def _summ(df: pd.DataFrame) -> Dict[str, float]:
    if df is None or len(df) == 0:
        return {"Recall@1": 0.0, "Recall@5": 0.0, "Recall@10": 0.0, "MRR@10": 0.0, "nDCG@10": 0.0}
    return {
        "Recall@1": float(df["is_hit_at_1"].mean()),
        "Recall@5": float(df["is_hit_at_5"].mean()),
        "Recall@10": float(df["is_hit_at_10"].mean()),
        "MRR@10": float(df["mrr@10"].mean()),
        "nDCG@10": float(df["ndcg@10"].mean()),
    }

s4b_all: List[pd.DataFrame] = []
s7a_all: List[pd.DataFrame] = []
summary_rows: List[Dict[str, Any]] = []

for (ds_tag, df, retr, qur_df, gold_col) in datasets:
    if df is None or len(df) == 0:
        print(f"[SKIP] {ds_tag}: empty dataset")
        continue

    print(f"\n=== Running gold-informed variants on: {ds_tag} (N={len(df)}) ===")

    s4b_df = evaluate_dataset(
        f"S4b_rrf_best ({ds_tag})",
        df,
        lambda q, g: system_s4b_rrf_best(
            retr, q, g, qur_df,
            top_n=TOP_N,
            rrf_k=RRF_K_DEFAULT,
            rewrite_weight=REWRITE_WEIGHT,
            max_rewrites=MAX_REWRITES,
            jaccard_min=REWRITE_JACCARD_MIN,
        ),
        gold_col=gold_col,
        dataset_tag=ds_tag,
    )
    s4b_all.append(s4b_df)

    s7a_df = evaluate_dataset(
        f"S7a_rerank_best ({ds_tag})",
        df,
        lambda q, g: system_s7a_rerank_best(
            retr, q, g, qur_df,
            top_n=TOP_N,
            max_rewrites=MAX_REWRITES,
            jaccard_min=REWRITE_JACCARD_MIN,
        ),
        gold_col=gold_col,
        dataset_tag=ds_tag,
    )
    s7a_all.append(s7a_df)

    summary_rows.append({"dataset": ds_tag, "system": "S4b_rrf_best", **_summ(s4b_df)})
    summary_rows.append({"dataset": ds_tag, "system": "S7a_rerank_best", **_summ(s7a_df)})

assert len(s4b_all) > 0 and len(s7a_all) > 0, "No results produced — check inputs."

s4b_combined = pd.concat(s4b_all, ignore_index=True)
s7a_combined = pd.concat(s7a_all, ignore_index=True)

out_s4b = os.path.join(ABLATION_OUTPUT_DIR, "S4b_rrf_best_ALL.csv")
out_s7a = os.path.join(ABLATION_OUTPUT_DIR, "S7a_rerank_best_ALL.csv")

s4b_combined.to_csv(out_s4b, index=False, encoding="utf-8")
s7a_combined.to_csv(out_s7a, index=False, encoding="utf-8")

print("\n[OK] Wrote:")
print(" -", out_s4b)
print(" -", out_s7a)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
